# Clase 1 · Ingesta y capa Bronze

Vamos a generar cuatro fuentes, inspeccionar sus formatos y materializarlas como tablas Delta Bronze. Bronze conserva el dato recibido: los problemas se identifican, pero se corrigen recién en Silver.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_data

In [ ]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
data = build_course_data(spark, config)
paths = write_landing_files(data, config)
paths

## 1. Leer no es todavía transformar

Inspeccioná los cuatro orígenes. Compará el esquema inferido del CSV con el esquema físico de Parquet. ¿Por qué `amount` termina como texto?

In [ ]:
customers_raw = spark.read.option("header", True).csv(paths["customers_csv"])
transactions_raw = spark.read.option("header", True).csv(paths["transactions_csv"])
products_raw = spark.read.parquet(paths["products_parquet"])
events_raw = spark.read.json(paths["events_json"])

for name, frame in {"customers": customers_raw, "transactions": transactions_raw, "products": products_raw, "events": events_raw}.items():
    print(f"\n--- {name} ---")
    frame.printSchema()

### Experimento: inferencia de tipos

Volvé a leer transacciones con `inferSchema=true`. Registrá cuánto tarda y verificá si resuelve correctamente `amount`. Explicá por qué inferir implica trabajo adicional y puede producir decisiones inestables.

In [ ]:
transactions_inferred = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["transactions_csv"]))
transactions_inferred.printSchema()
display(transactions_inferred.limit(10))

## 2. Metadatos de ingesta

Bronze agrega trazabilidad sin modificar la semántica del dato de origen. `_source_file` identifica el archivo físico y `_source_path` el directorio de entrada.

In [ ]:
from pyspark.sql import functions as F

def with_ingestion_metadata(frame, source_name, source_path):
    return (frame
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
        .withColumn("_source", F.lit(source_name))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_ingested_at", F.current_timestamp()))

bronze = {
    "customers": with_ingestion_metadata(customers_raw, "customers_csv", paths["customers_csv"]),
    "products": with_ingestion_metadata(products_raw, "products_parquet", paths["products_parquet"]),
    "transactions": with_ingestion_metadata(transactions_raw, "transactions_csv", paths["transactions_csv"]),
    "events": with_ingestion_metadata(events_raw, "events_json", paths["events_json"]),
}

In [ ]:
for name, frame in bronze.items():
    table = f"{config.catalog}.{config.schema}.bronze_{name}"
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table)
    print(f"{table}: {spark.table(table).count():,} filas")

## 3. Diagnóstico inicial de calidad

No limpies todavía. Medí cuántos identificadores están duplicados y cuántos importes no pueden convertirse a número.

In [ ]:
tx = spark.table(f"{config.catalog}.{config.schema}.bronze_transactions")
quality_summary = tx.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    F.sum(F.expr("CASE WHEN try_cast(amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END")).alias("invalid_amounts"),
)
display(quality_summary)

## 4. Delta y el plan de ejecución

Ejecutá `DESCRIBE DETAIL`, `DESCRIBE HISTORY` y `EXPLAIN FORMATTED` sobre una tabla. Identificá qué información agrega Delta respecto de un directorio Parquet.

In [ ]:
table_name = f"{config.catalog}.{config.schema}.bronze_transactions"
display(spark.sql(f"DESCRIBE DETAIL {table_name}"))
display(spark.sql(f"DESCRIBE HISTORY {table_name}"))
spark.sql(f"EXPLAIN FORMATTED SELECT payment_channel, count(*) FROM {table_name} GROUP BY payment_channel").show(truncate=False)

## Entrega breve

Dejá ejecutadas en este notebook las salidas de las cuatro tablas y del diagnóstico de calidad. Las respuestas escritas van en el `README.md` de tu entrega: (1) tres observaciones sobre CSV/JSON, Parquet y Delta; y (2) una explicación de dónde aparece volumen, velocidad, variedad, veracidad y valor en este caso. El formato completo está en la sección **Formato de entrega** del README de la práctica.